# Guidance-Gap Ablation Runner (Exp #6)

Sweeps γ₂ for SDXL standard vs Z-Sampling (264 images). Set **Runtime → T4 GPU** before running.

## Cell 1 — Mount Drive and clone repo

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, os
from getpass import getpass

REPO_DIR = "/content/generative-modelling"
BRANCH = "radu-exp-6"

if not os.path.exists(REPO_DIR):
    try:
        print("Attempting to clone repository publicly...")
        subprocess.run(
            ["git", "clone", "https://github.com/viftode4/generative-modelling.git", REPO_DIR],
            check=True,
        )
    except subprocess.CalledProcessError:
        print("\nClone failed. The repository is private.")
        token = getpass("Paste your GitHub Personal Access Token (PAT): ")
        clone_url = f"https://{token}@github.com/viftode4/generative-modelling.git"
        subprocess.run(["git", "clone", clone_url, REPO_DIR], check=True)

subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)
print(f"Repo ready at {REPO_DIR} on branch {BRANCH}")

ModuleNotFoundError: No module named 'google'

## Cell 2 — Install deps
Scoring deps not included — scoring runs locally.

In [ ]:
!pip install -q \
  diffusers==0.37.1 transformers==5.8.1 accelerate safetensors einops torchsde \
  'huggingface_hub>=0.20' 'numpy>=2,<3' tqdm Pillow

import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
assert torch.cuda.is_available(), 'Set Runtime -> Change runtime type -> T4 GPU!'

## Cell 3 — Smoke test (1 prompt, 3 variants)
Sanity check before the full run.

In [ ]:
import os
os.chdir(REPO_DIR)
SMOKE_OUT = "/content/drive/MyDrive/radu_guidance_gap_smoke"

!PYTHONPATH=src python scripts/run_radu_guidance_gap.py \
    --prompts experiments/prompts/pickapic_full.md \
    --seeds 42 --limit 1 --gamma2-sweep 0.0 5.5 \
    --device cuda --size 1024 --steps 50 --lambda-step 49 --gamma1 5.5 \
    --out {SMOKE_OUT}

## Cell 4 — Verify smoke images

In [ ]:
import json
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

rows = [json.loads(l) for l in (Path(SMOKE_OUT) / 'manifest.jsonl').read_text().splitlines()]
print(f"{len(rows)} rows in manifest")

first = rows[0]['prompt_id']
grp = [r for r in rows if r['prompt_id'] == first]
std = next(r for r in grp if r['method'] == 'standard')
zs = [r for r in grp if r['method'] != 'standard'][:2]
show = [std] + zs

fig, axes = plt.subplots(1, len(show), figsize=(5 * len(show), 5))
fig.suptitle(std['prompt'], fontsize=10)
for ax, r in zip(axes, show):
    ax.imshow(Image.open(r['image_path']))
    label = 'Standard' if r['method'] == 'standard' else f"Z γ2={r['gamma_2']}"
    ax.set_title(label)
    ax.axis('off')
plt.tight_layout(); plt.show()

## Cell 5 — Full run (264 images, ~6–12h)
Resumable — rerun the cell after a disconnect and already-generated images are skipped.

In [ ]:
%cd /content/generative-modelling
FULL_OUT = "/content/drive/MyDrive/radu_guidance_gap_full"

!PYTHONPATH=src python scripts/run_radu_guidance_gap.py \
    --prompts experiments/prompts/pickapic_full.md \
    --seeds 42 --limit 24 \
    --device cuda --size 1024 --steps 50 --lambda-step 49 --gamma1 5.5 \
    --out {FULL_OUT}

## Checkpoint

In [ ]:
from pathlib import Path
imgs = list((Path('/content/drive/MyDrive/radu_guidance_gap_full') / 'images').glob('*.png'))
std = [p for p in imgs if '_standard' in p.name]
print(f"Images on Drive: {len(imgs)} / 264   (standard baselines: {len(std)} / 24)")
print('COMPLETE' if len(imgs) >= 264 else 'not finished — rerun Cell 5 in a new session')

## Cell 6 — Score locally
Download the `radu_guidance_gap_full` Drive folder to the Mac, then run the local scoring scripts.

In [ ]:
# Optional: one tarball to download from the Files panel instead of syncing Drive.
!tar czf /content/radu_guidance_gap_full.tgz -C /content/drive/MyDrive radu_guidance_gap_full
!ls -lh /content/radu_guidance_gap_full.tgz